# How the heat-wave detector works (one event)

A walk through `heatwave_ic/detect.py` on the most severe event of the 2026
catalogue, the 12 June to 3 July spell over central Europe (rank 1 in
`data/heatwave_events_europe_2026.csv`). Every step runs the same function the
catalogue script calls, so what this notebook shows is what the paper's
methods section describes.

The pipeline, in the order the code runs it:

1. **Threshold.** A calendar-day 90th percentile of daily maximum temperature,
   per cell, from the 1991-2020 baseline.
2. **Exceedance.** Mark each land cell-day whose maximum sits above its own
   threshold.
3. **Area filter, day by day.** Drop every patch of hot cells smaller than
   50,000 km$^2$ on that day.
4. **Persistence.** Keep a cell-day only if it belongs to a run of at least
   three consecutive surviving days at that cell.
5. **Labelling.** Group what survives into space-time connected components.
   Each component is an event.
6. **Attributes.** Severity, dates, peak day, centroid, and the rest of the
   table row.

The order of steps 3 and 4 is part of the definition. The published CTX90pct
index applies persistence to the raw exceedance; here it is applied to the
field the area filter leaves behind. Swapping the two gives a different field,
and step 4 measures how different.

## Setup

In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import ndimage

YEAR = 2026
A_MIN, MIN_DAYS, Q = 50_000.0, 3, 0.90       # the reference configuration
# Downloads and thresholds live in the hwcat working folder; the published
# tables live in paper/results. Set HWCAT_WORKDIR to point elsewhere.
WORKDIR = Path(os.environ.get("HWCAT_WORKDIR", "../hwcat-data"))
RESULTS = Path("../paper/results")

# hwcat.plots calls mpl_apply() on import, which is where the project rcParams
# come from. Do not set rcParams here.
from hwcat import detect as D, plots as P  # noqa: E402

BORDERS = Path(P.__file__).parent / "data" / "borders_europe_ne50m.json"
rings = json.loads(BORDERS.read_text()) if BORDERS.exists() else []


def draw_borders(ax):
    for ring in rings:
        ax.plot([p[0] for p in ring], [p[1] for p in ring],
                color="#333333", linewidth=0.3, zorder=3)
    ax.set_xlim(lon.min(), lon.max())      # rings run past the domain
    ax.set_ylim(lat.min(), lat.max())


def show(ax, field, lat, lon, **kw):
    # imshow with the grid's own extent, so lat/lon annotate on top directly.
    return ax.imshow(field, origin="upper", interpolation="nearest",
                     aspect="auto",
                     extent=[lon.min(), lon.max(), lat.min(), lat.max()], **kw)

## The data and the event

The target season is daily maximum 2 m temperature from ERA5, 1 May to 5
September 2026, on the native 0.25$^\circ$ grid. The threshold file is the
cached output of `calendar_day_percentile`, one field per calendar day.

Grid cells shrink towards the pole, so every area in this notebook is a true
spherical area from `cell_area_km2`, never a cell count.

In [ ]:
target = D.open_daily_tmax([WORKDIR / "era5" / f"europe_tmax_{YEAR}.nc"])
# Built by `hwcat detect --year 2026` if it is not there yet.
thresh = xr.open_dataarray(
    WORKDIR / "thresholds" / "heatwave_threshold_europe_1991_2020_p90_m04-10.nc").load()
lat, lon = target["latitude"].values, target["longitude"].values
land = D.land_mask_for(lat, lon)
area = D.cell_area_km2(lat, lon)
time = pd.DatetimeIndex(target["time"].values)

print(f"{time.size} days, {time[0]:%d %b} to {time[-1]:%d %b} {YEAR}; "
      f"grid {lat.size} x {lon.size}")
print(f"land cells: {land.sum()} of {land.size} ({100 * land.mean():.0f}%)")
print(f"cell area: {area[0, 0]:,.0f} km2 at {lat[0]:.0f}N, "
      f"{area[-1, 0]:,.0f} km2 at {lat[-1]:.0f}N "
      f"(ratio {area[0, 0] / area[-1, 0]:.2f})")

In [ ]:
events = pd.read_csv(RESULTS / f"heatwave_events_europe_{YEAR}.csv",
                     parse_dates=["start", "peak", "end"])
ev = events.iloc[0]
ev

## Step 1: the threshold at one cell

Each cell has its own threshold for each calendar day. For a target day $d$
the code pools every baseline day whose day-of-year lies within 7 days of $d$,
over all 30 years, and takes the 90th percentile of the pool. That is 450
samples when the whole window lies inside May to September, and fewer at the
edges of the season, because the baseline was only retrieved for those months.

The cell below rebuilds the threshold for the event's peak day at the grid
cell nearest its centroid, straight from the baseline files, and checks it
against the cached field. It also counts the pool on 1 May, where the window
is cut off.

In [ ]:
i = int(np.abs(lat - ev.centroid_lat).argmin())
j = int(np.abs(lon - ev.centroid_lon).argmin())
peak_doy = int(ev.peak.dayofyear)
print(f"cell {lat[i]:.2f}N {lon[j]:.2f}E, peak day {ev.peak:%d %b} "
      f"(day-of-year {peak_doy})")

# One point across the 30 baseline files. open_mfdataset is lazy, so this
# reads a single column rather than the 770 MB the catalogue script loads.
files = [WORKDIR / "era5" / f"europe_tmax_{y}.nc" for y in range(1991, 2021)]
ds = xr.open_mfdataset([str(f) for f in files], combine="by_coords")
var = next(v for v in D._TMAX_NAMES if v in ds.data_vars)
tdim = next(t for t in D._TIME_NAMES if t in ds[var].dims)
latn = "latitude" if "latitude" in ds[var].dims else "lat"
lonn = "longitude" if "longitude" in ds[var].dims else "lon"
point = ds[var].isel({latn: i, lonn: j}).load()
if float(point.max()) > 100:            # kelvin
    point = point - 273.15
base_doy = point[tdim].dt.dayofyear.values


def pooled_p90(doy):
    delta = np.abs(base_doy - doy)
    delta = np.minimum(delta, 366 - delta)
    pool = point.values[delta <= 7]
    return pool.size, float(np.quantile(pool, Q))


n, t90_by_hand = pooled_p90(peak_doy)
t90_cached = float(thresh.sel(
    dayofyear=peak_doy).isel(latitude=i, longitude=j))
print(f"{n} pooled samples -> T90 = {t90_by_hand:.2f} C by hand, "
      f"{t90_cached:.2f} C cached")
assert abs(t90_by_hand - t90_cached) < 0.05

n_may1, _ = pooled_p90(int(pd.Timestamp(f"{YEAR}-05-01").dayofyear))
print(
    f"on 1 May the pool holds only {n_may1} samples: the window is truncated")

The plot shows the 2026 season at that cell against its moving threshold. Every
red day is an exceedance; the grey band is the event as the catalogue finally
records it. Notice that exceedance at one cell is neither necessary nor
sufficient for the cell to be inside the event: the event is defined by the
steps that follow, not by this cell alone.

In [ ]:
tmax_cell = target.isel(latitude=i, longitude=j).values
t90_cell = thresh.sel(dayofyear=target["time"].dt.dayofyear).isel(
    latitude=i, longitude=j).values

fig, ax = plt.subplots(figsize=(10, 3.6))
ax.axvspan(ev.start, ev.end, color="0.85", label="event (rank 1)")
ax.plot(time, t90_cell, color="black", linewidth=1, label="calendar-day T90")
ax.plot(time, tmax_cell, color="tab:blue",
        linewidth=1, label="daily Tmax 2026")
ax.fill_between(time, t90_cell, tmax_cell, where=tmax_cell > t90_cell,
                color="tab:red", alpha=0.5, label="exceedance")
ax.axvline(ev.peak, color="tab:red", linewidth=0.8, linestyle="--")
ax.set(ylabel="temperature (C)",
       title=f"{lat[i]:.2f}N {lon[j]:.2f}E: daily maximum against its own threshold")
ax.legend(loc="upper left", fontsize=8, ncol=4)
plt.show()

## Step 2: exceedance on the peak day

`exceedance` compares every cell-day with its threshold and returns a boolean
field. Ocean cells are removed with the land mask. By construction about a
tenth of all land cell-days sit above a 90th percentile, so even the map of one
day is speckled with isolated hot cells that have nothing to do with the
event.

In [ ]:
exc = D.exceedance(target, thresh).values & land[None, :, :]
k = int(np.flatnonzero(time == ev.peak)[0])
hot = exc[k]
print(f"{ev.peak:%d %b}: {hot.sum()} of {land.sum()} land cells "
      f"({100 * hot.sum() / land.sum():.0f}%) above their threshold")
print(f"whole season: {100 * exc[:, land].mean():.1f}% of land cell-days")

## Step 3: the area filter, day by day

`spatially_coherent` labels the hot cells of each day into 8-connected
patches, sums the spherical area of each patch, and drops any patch under
$A_{\min}$. This is the step that stops scattered exceedances from bridging
unrelated blobs during the space-time labelling. The two maps are the peak day
before and after it.

In [ ]:
labels2d, n2d = ndimage.label(
    hot, structure=ndimage.generate_binary_structure(2, 2))
patch_area = ndimage.sum_labels(area, labels2d, index=np.arange(1, n2d + 1))
big = patch_area >= A_MIN
coh = D.spatially_coherent(exc, area, A_MIN)
print(f"{n2d} patches on {ev.peak:%d %b}; {big.sum()} reach {A_MIN:,.0f} km2 "
      f"and hold {patch_area[big].sum() / patch_area.sum():.0%} of the hot area")
print(f"largest patch: {patch_area.max():,.0f} km2")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
for ax, field, title in ((axes[0], hot, "above threshold"),
                         (axes[1], coh[k], f"patches of at least {A_MIN:,.0f} km$^2$")):
    show(ax, np.where(field, 1.0, np.nan), lat,
         lon, cmap="Reds", vmin=0, vmax=1.3)
    show(ax, np.where(land, np.nan, 0.5), lat,
         lon, cmap="Blues", vmin=0, vmax=1)
    draw_borders(ax)
    ax.set(title=f"{ev.peak:%d %b %Y}: {title}", xlabel="longitude")
axes[0].set_ylabel("latitude")
plt.show()

## Step 4: persistence

`sustained` is a binary opening along the time axis with a structuring element
three days long. The erosion deletes every day that is not the middle of a
three-day run, and the dilation restores the runs that survive, so a cell keeps
a day only if it belongs to a run of at least three.

Because the opening runs after the area filter, a run is broken whenever the
cell's patch falls below $A_{\min}$ for a day. The reverse order would break
patches instead: persistence thins a patch, and the thinned patch can then fall
under $A_{\min}$. Neither order is stricter than the other everywhere, so the
choice has to be stated. The strip tracks one cell through the three stages,
and the count afterwards is the difference the order makes across the season.

In [ ]:
sus = D.sustained(coh, MIN_DAYS)
alt = D.spatially_coherent(D.sustained(exc, MIN_DAYS), area, A_MIN)

print(f"land cell-days above threshold : {exc.sum():>8,}")
print(f"after the area filter          : {coh.sum():>8,}")
print(f"after persistence (pipeline)   : {sus.sum():>8,}")
print(f"persistence first, area second : {alt.sum():>8,}  "
      f"({100 * (alt.sum() - sus.sum()) / sus.sum():+.1f}%)")

win = slice(int(np.flatnonzero(time == ev.start - pd.Timedelta(days=5))[0]),
            int(np.flatnonzero(time == ev.end + pd.Timedelta(days=5))[0]) + 1)
strip = np.stack([exc[win, i, j], coh[win, i, j], sus[win, i, j]])
fig, ax = plt.subplots(figsize=(10, 1.8))
ax.imshow(strip, cmap="Greys", aspect="auto", interpolation="nearest")
ax.set_yticks(range(3))
ax.set_yticklabels(["above threshold", "in a large patch", "3-day run"])
ticks = np.arange(0, strip.shape[1], 5)
ax.set_xticks(ticks)
ax.set_xticklabels([f"{d:%d %b}" for d in time[win][ticks]], fontsize=8)
ax.set_title(f"cell {lat[i]:.2f}N {lon[j]:.2f}E through the three stages")
plt.show()

## Step 5: labelling the events

`detect` runs the four stages above and then calls `scipy.ndimage.label` on
the (time, lat, lon) field with a structure that joins each cell to its eight
spatial neighbours on the same day and to itself on the day before and after.
`event_table` then keeps a component if its largest daily area reaches
$A_{\min}$, which can fail only because the persistence step eroded cells out
of a patch that had passed the area filter, and builds one row per event.

In [ ]:
det = D.detect(target, thresh, land_mask=land,
               min_days=MIN_DAYS, min_area_km2=A_MIN)
assert (det.mask == sus).all(), "the stages above are not what detect() ran"

table = D.event_table(det, target, min_area_km2=A_MIN)
row = table.iloc[0]
eid = int(row.event_id)
assert row.start == ev.start.date() and row.end == ev.end.date()
print(f"{det.n_events} components, {len(table)} pass the peak-area check; "
      f"component {eid} is rank 1")

inside = det.labels == eid
days_in = inside.sum(axis=0)
daily_area = (inside * area[None]).sum(axis=(1, 2))

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(12, 4.2),
                               gridspec_kw={"width_ratios": [1.5, 1]})
show(ax0, np.where(days_in > 0, days_in, np.nan), lat, lon, cmap="inferno_r")
show(ax0, np.where(land, np.nan, 0.5), lat, lon, cmap="Blues", vmin=0, vmax=1)
draw_borders(ax0)
ax0.scatter(ev.centroid_lon, ev.centroid_lat, s=60, facecolor="white",
            edgecolor="black", zorder=5, label="centroid (peak day)")
ax0.scatter(ev.peak_tmax_lon, ev.peak_tmax_lat, s=60, marker="^",
            facecolor="tab:blue", edgecolor="black", zorder=5, label="hottest cell")
ax0.legend(loc="lower left", fontsize=8)
ax0.set(title=f"event {eid}: days each cell spent inside it",
        xlabel="longitude", ylabel="latitude")
ax1.bar(time, daily_area / 1e6, width=1, color="tab:red")
ax1.axvline(ev.peak, color="black", linestyle="--",
            linewidth=0.8, label="peak day")
ax1.set_xlim(ev.start - pd.Timedelta(days=3), ev.end + pd.Timedelta(days=3))
ax1.set(ylabel="area inside the event (10$^6$ km$^2$)", title="daily extent")
ax1.xaxis.set_major_locator(mdates.DayLocator(interval=5))
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax1.tick_params(axis="x", labelsize=8)
ax1.legend(fontsize=8)
plt.show()

The footprint runs from Iberia to the Baltic, and the daily extent has a
shoulder around 17 June before the peak ten days later. The labeller keeps the
two phases together because hot cells persist somewhere in the footprint on
every day in between; it never asks whether they are the same weather system.
That is the sense in which an event here is a connected object, not a
synoptic judgement, and it is why the sensitivity test at the end matters.

## Step 6: the attributes, computed by hand

Severity is the exceedance anomaly summed over every cell-day in the event and
weighted by cell area, in $^\circ$C km$^2$ day. The peak day is the day with
the largest mean anomaly over the event's cells (an unweighted mean), and the
centroid is the area-weighted mean position of the cells on that day. Each is
recomputed below from the label field and checked against the table row.

In [ ]:
severity = float((det.anomaly * area[None])[inside].sum())
n_by_day = inside.sum(axis=(1, 2))
mean_anom = np.where(n_by_day > 0,
                     (det.anomaly * inside).sum(axis=(1, 2)) /
                     np.maximum(n_by_day, 1),
                     np.nan)
t_peak = int(np.nanargmax(mean_anom))
w = inside[t_peak] * area
c_lat = float((w.sum(axis=1) * lat).sum() / w.sum())
c_lon = float((w.sum(axis=0) * lon).sum() / w.sum())

checks = pd.DataFrame({
    "by hand": [severity, str(time[t_peak].date()), c_lat, c_lon,
                int(inside.any(axis=(1, 2)).sum()), float(daily_area.max())],
    "table":   [row.severity_C_km2_day, str(row.peak), row.centroid_lat,
                row.centroid_lon, row.duration_days, row.peak_area_km2],
}, index=["severity (C km2 day)", "peak day", "centroid lat", "centroid lon",
          "duration (days)", "peak area (km2)"])
assert abs(severity - row.severity_C_km2_day) < 1
assert time[t_peak].date() == row.peak
checks

## Step 7: what the choices do to this event

Two of the settings above are conventions: the percentile and $A_{\min}$.
`scripts/catalogue_sensitivity.py` reruns the detector for six combinations and
matches the reference events into each run by overlapping dates and nearest
centroid. The table pulls out our event, reference rank 1, and reference rank
4, the 30 June to 24 July spell over central France that follows it.

At $A_{\min} = 25{,}000$ km$^2$ the two are one 44-day object: the smaller
patches that survive at that setting bridge the gap between them. At the 95th
percentile the event keeps its rank but loses a quarter of its duration and
40% of its severity, because the threshold is higher and the anomaly above it
correspondingly smaller.

In [ ]:
sens = pd.read_csv(RESULTS / f"catalogue_sensitivity_europe_{YEAR}.csv",
                   parse_dates=["start", "end"])
cols = ["ref_rank", "percentile", "min_area_km2", "rank", "start", "end",
        "duration_days", "severity_C_km2_day"]
view = (sens[sens.ref_rank.isin([1, 4])][cols]
        .assign(start=lambda d: d.start.dt.strftime("%d %b"),
                end=lambda d: d.end.dt.strftime("%d %b"),
                severity_C_km2_day=lambda d: (d.severity_C_km2_day / 1e6).round(1))
        .rename(columns={"severity_C_km2_day": "severity (1e6 C km2 day)"})
        .sort_values(["ref_rank", "percentile", "min_area_km2"])
        .set_index(["ref_rank", "percentile", "min_area_km2"]))
view

## What to take from this

- An event is a connected component of cell-days that passed two filters. Its
  extent, duration, and severity all depend on the order and the settings of
  those filters, which is why the paper treats the percentile and $A_{\min}$ as
  free parameters rather than as settled choices.
- The three-day rule is evaluated after the area filter, not on the raw
  exceedance as in the published CTX90pct. The two orders disagree on about
  2% of cell-days, so the order belongs in the methods.
- The threshold pool is 450 samples only away from the season edges. Events
  that start on 1 May or end on the last day of the data are cut off by the
  window, not by the weather.